## LoCaLS: running examples

The workflow is:

1. load and validate the example inputs;
2. identify observed, latent, and selection variables;
3. choose one observed target;
4. learn a local PAG from observations;
5. oracle learning.



## 1. Setup


In [47]:
from pathlib import Path

import pandas as pd

from method.locaLS import data_LoCaLS, oracle_LoCaLS
from Utils.util_tools import local_mark_evaluation

pd.options.display.float_format = '{:.4f}'.format

## 2. Load and prepare the example data

The bundled files have distinct roles:

- `dag_mat.csv`: the known 100-node DAG used by the oracle CI test;
- `all_data.csv`: 1,000 simulated continuous observations for all DAG nodes; and
- `pag_mat.csv`: the ground-truth PAG over observed variables.

Variables beginning with `L` are treated as latent and variables beginning with `S` as selection variables. Neither group is passed to the data-driven learner.

In [48]:
data_dir = Path('example_data')

dag = pd.read_csv(data_dir / 'dag_mat.csv', index_col=0)
all_data = pd.read_csv(data_dir / 'all_data.csv')
true_pag = pd.read_csv(data_dir / 'pag_mat.csv', index_col=0)

latent_nodes = [node for node in dag.columns if node.startswith('L')]
selection_nodes = [node for node in dag.columns if node.startswith('S')]
hidden_nodes = set(latent_nodes) | set(selection_nodes)
observed_vars = [node for node in dag.columns if node not in hidden_nodes]
observed_data = all_data.loc[:, observed_vars]

assert dag.index.equals(dag.columns)
assert observed_data.columns.equals(true_pag.columns)
assert true_pag.index.equals(true_pag.columns)

In [49]:
pd.Series(
    {
        'samples': len(observed_data),
        'DAG nodes': len(dag),
        'observed variables': len(observed_vars),
        'latent variables': len(latent_nodes),
        'selection variables': len(selection_nodes),
    },
    name='example data',
).to_frame()

,example data
samples,1000
DAG nodes,100
observed variables,90
latent variables,5
selection variables,5


## 3. Choose an observed target


In [50]:
target_degrees = true_pag.ne(0).sum(axis=1)
target = target_degrees.idxmax()

pd.Series(
    {
        'target': target,
        'degree in true PAG': int(target_degrees[target]),
    },
    name='selected target',
).to_frame()

,selected target
target,V82
degree in true PAG,6


## 4. Learn from observational data

`data_LoCaLS` uses Fisher-Z CI tests. Here, `alpha` is the CI-test significance level, `max_K` caps the conditioning-set size. 

In [51]:
data_result = data_LoCaLS(
    observed_data,
    target=target,
    alpha=0.01,
    max_K=5,
)
data_pag = data_result['PAG.DataFrame']

data_metrics = local_mark_evaluation(data_pag, true_pag, target)
data_summary = pd.Series(
    {
        **data_metrics,
        'CI tests': data_result['CI_num'],
        'runtime (s)': data_result['runtime_sec'],
    },
    name='data-driven',
)
data_summary.to_frame()

,data-driven
Local-SHD,1.0000
Mark-Precision,0.9167
Mark-Recall,0.9167
Mark-F1,0.9167
CI tests,997.0000
runtime (s),0.0718


## 5. Oracle learning

The oracle learner replaces finite-sample CI decisions with oracle-separation queries in the known DAG. It receives the latent and selection node lists explicitly.

In [52]:
oracle_result = oracle_LoCaLS(
    dag,
    target=target,
    latent_nodes=latent_nodes,
    selection_bias_nodes=selection_nodes,
)
oracle_pag = oracle_result['PAG.DataFrame']

oracle_metrics = local_mark_evaluation(oracle_pag, true_pag, target)
oracle_summary = pd.Series(
    {
        **oracle_metrics,
        'CI tests': oracle_result['CI_num'],
        'runtime (s)': oracle_result['runtime_sec'],
    },
    name='oracle',
)

comparison = pd.concat([data_summary, oracle_summary], axis=1)
comparison

,data-driven,oracle
Local-SHD,1.0000,0.0000
Mark-Precision,0.9167,1.0000
Mark-Recall,0.9167,1.0000
Mark-F1,0.9167,1.0000
CI tests,997.0000,1679.0000
runtime (s),0.0718,0.5630
